
# Lab Sheet 4 – Fine-Tuning an LLM using LoRA/QLoRA

## Objective
To fine-tune a small instruction-tuned Large Language Model (LLM) using Low-Rank Adaptation (LoRA) or Quantized LoRA (QLoRA) and evaluate its performance on domain-specific questions.


In [2]:

%pip install -q transformers datasets peft trl accelerate bitsandbytes


In [1]:

import torch

# Check GPU availability
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)

    print("GPU Name:", gpu_name)
    print(f"Total GPU Memory: {gpu_memory:.2f} GB")
else:
    print("No GPU detected. Please enable GPU in Colab.")


GPU Available: True
GPU Name: Tesla T4
Total GPU Memory: 14.56 GB


In [4]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load pretrained model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully!")
print("Model:", model_name)
print("Device:", next(model.parameters()).device)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully!
Model: Qwen/Qwen2.5-0.5B-Instruct
Device: cuda:0


In [5]:

from datasets import Dataset

# Custom Computer Shop Customer Support Dataset
qa_pairs = [
    ("What is your shop name?", "Our shop is called TechZone Computers."),
    ("Where is TechZone Computers located?", "TechZone Computers is located in Colombo, Sri Lanka."),
    ("What are your opening hours?", "We are open Monday to Saturday, from 9 AM to 6 PM."),
    ("Are you open on Sundays?", "No, our shop is closed on Sundays."),
    ("What products do you sell?", "We sell laptops, desktop computers, monitors, keyboards, mice, and computer accessories."),
    ("Do you sell gaming laptops?", "Yes, we offer gaming laptops starting from LKR 250,000."),
    ("What is the price of a basic laptop?", "Our basic laptops start from LKR 120,000."),
    ("Do you sell desktop computers?", "Yes, desktop computers are available starting from LKR 100,000."),
    ("Do you have student laptops?", "Yes, we offer affordable laptops suitable for students."),
    ("Do you sell refurbished laptops?", "Yes, we sell tested refurbished laptops with a 6-month warranty."),
    ("Do your laptops come with a warranty?", "Yes, new laptops include a 12-month warranty."),
    ("What warranty do refurbished laptops have?", "Refurbished laptops include a 6-month warranty."),
    ("Can I return a product?", "Yes, eligible products can be returned within 7 days with the original receipt."),
    ("Do you offer refunds?", "Yes, refunds are available for eligible returns under our return policy."),
    ("Can I exchange a product?", "Yes, eligible products can be exchanged within 7 days."),
    ("Do you repair laptops?", "Yes, we provide laptop repair services."),
    ("Do you repair desktop computers?", "Yes, we repair desktop computers and diagnose hardware problems."),
    ("Can you replace a laptop battery?", "Yes, we provide laptop battery replacement services."),
    ("Do you repair broken laptop screens?", "Yes, we offer laptop screen replacement services."),
    ("Can you upgrade my laptop RAM?", "Yes, we offer RAM upgrades for compatible laptops."),
    ("Can you install an SSD?", "Yes, we provide SSD installation and upgrade services."),
    ("Do you sell keyboards?", "Yes, we sell wired and wireless keyboards."),
    ("Do you sell wireless mice?", "Yes, wireless mice are available."),
    ("Do you sell monitors?", "Yes, we offer monitors in different sizes."),
    ("Do you sell printers?", "Yes, we sell printers for home and office use."),
    ("Do you provide home delivery?", "Yes, we provide delivery across Sri Lanka."),
    ("How long does delivery take?", "Delivery usually takes 2 to 5 working days."),
    ("How much is the delivery fee?", "Delivery fees depend on the customer's location."),
    ("What payment methods do you accept?", "We accept cash, debit cards, credit cards, and bank transfers."),
    ("Can I pay by bank transfer?", "Yes, bank transfers are accepted."),
    ("Do you offer installment payments?", "Yes, installment payments are available for selected products."),
    ("Do you offer student discounts?", "Yes, selected products have special student discounts."),
    ("Can I order products online?", "Yes, customers can place orders through our online store."),
    ("How can I contact customer support?", "You can contact TechZone Computers by phone, email, or by visiting our shop."),
    ("Do you provide technical advice?", "Yes, our team can help customers choose suitable computers and accessories.")
]

# Convert to a Hugging Face Dataset
dataset = Dataset.from_list([
    {
        "question": question,
        "answer": answer
    }
    for question, answer in qa_pairs
])

print("Dataset created successfully!")
print("Total Q&A pairs:", len(dataset))

print("\nFirst 3 examples:")
for i in range(3):
    print(f"\nQuestion: {dataset[i]['question']}")
    print(f"Answer: {dataset[i]['answer']}")


Dataset created successfully!
Total Q&A pairs: 35

First 3 examples:

Question: What is your shop name?
Answer: Our shop is called TechZone Computers.

Question: Where is TechZone Computers located?
Answer: TechZone Computers is located in Colombo, Sri Lanka.

Question: What are your opening hours?
Answer: We are open Monday to Saturday, from 9 AM to 6 PM.


In [6]:

from peft import LoraConfig, get_peft_model

# Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to the pretrained model
model = get_peft_model(model, lora_config)

print("LoRA configured successfully!")
model.print_trainable_parameters()


LoRA configured successfully!
trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [7]:

# Convert Q&A pairs into chat-formatted training examples
def format_example(example):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful customer support assistant for TechZone Computers."
        },
        {
            "role": "user",
            "content": example["question"]
        },
        {
            "role": "assistant",
            "content": example["answer"]
        }
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
    }

training_dataset = dataset.map(
    format_example,
    remove_columns=dataset.column_names
)

print("Dataset formatted successfully!")
print("Total training examples:", len(training_dataset))
print("\nFirst formatted example:")
print(training_dataset[0]["text"])


Map:   0%|          | 0/35 [00:00<?, ? examples/s]

Dataset formatted successfully!
Total training examples: 35

First formatted example:
<|im_start|>system
You are a helpful customer support assistant for TechZone Computers.<|im_end|>
<|im_start|>user
What is your shop name?<|im_end|>
<|im_start|>assistant
Our shop is called TechZone Computers.<|im_end|>



In [9]:

%pip install -q trl


In [10]:

from trl import SFTTrainer, SFTConfig

# Training configuration
training_args = SFTConfig(
    output_dir="./techzone_lora_output",
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_length=256,
    logging_steps=5,
    save_strategy="no",
    fp16=True,
    report_to="none",
    dataset_text_field="text",
    packing=False
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=training_dataset,
    processing_class=tokenizer
)

print("Trainer initialized successfully!")

# Start fine-tuning
trainer.train()

print("Fine-tuning completed successfully!")


Adding EOS to train dataset:   0%|          | 0/35 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/35 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/35 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/35 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/35 [00:00<?, ? examples/s]

Trainer initialized successfully!


Step,Training Loss
5,5.291104
10,4.065092
15,3.098644
20,2.471943
25,2.113581
30,1.728528
35,1.330603
40,1.092135
45,0.935031
50,0.865342


Fine-tuning completed successfully!


In [11]:

# Save the fine-tuned LoRA adapter
model.save_pretrained("techzone_lora_adapter")

# Save the tokenizer
tokenizer.save_pretrained("techzone_lora_adapter")

print("Fine-tuned LoRA adapter saved successfully!")
print("Folder: techzone_lora_adapter")


Fine-tuned LoRA adapter saved successfully!
Folder: techzone_lora_adapter


In [13]:

test_questions = [
    "What is your shop name?",
    "Where is TechZone Computers located?",
    "Do you repair laptops?",
    "Can I get my laptop screen replaced?",
    "How long will it take to deliver a gaming laptop?",
    "What warranty is available for a refurbished laptop?"
]

print("Model evaluation questions prepared!")
print("Total questions:", len(test_questions))

for i, question in enumerate(test_questions, 1):
    print(f"{i}. {question}")


Model evaluation questions prepared!
Total questions: 6
1. What is your shop name?
2. Where is TechZone Computers located?
3. Do you repair laptops?
4. Can I get my laptop screen replaced?
5. How long will it take to deliver a gaming laptop?
6. What warranty is available for a refurbished laptop?


In [14]:

# Generate a response to a customer question
def get_response(question):
    messages = [
        {
            "role": "system",
            "content": "You are a helpful customer support assistant for TechZone Computers."
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# Compare original and fine-tuned responses
model.eval()

comparison_results = []

for question in test_questions:

    # Original model: temporarily disable LoRA
    with model.disable_adapter():
        original_answer = get_response(question)

    # Fine-tuned model: LoRA enabled
    fine_tuned_answer = get_response(question)

    comparison_results.append({
        "Question": question,
        "Original": original_answer,
        "Fine-Tuned": fine_tuned_answer
    })

    print("\nQUESTION:", question)
    print("ORIGINAL MODEL:", original_answer)
    print("FINE-TUNED MODEL:", fine_tuned_answer)
    print("-" * 60)

print("\nModel comparison completed successfully!")



QUESTION: What is your shop name?
ORIGINAL MODEL: My shop name is TechZone Computers.
FINE-TUNED MODEL: TechZone Computers
------------------------------------------------------------

QUESTION: Where is TechZone Computers located?
ORIGINAL MODEL: TechZone Computers is located in the United States, specifically in the city of San Francisco. The company was founded in 1986 and has offices in various locations around the world, including San Francisco, Los Angeles, New York City, and London.
FINE-TUNED MODEL: TechZone Computers is located in the city centre of Sydney, Australia.
------------------------------------------------------------

QUESTION: Do you repair laptops?
ORIGINAL MODEL: I'm sorry, but I can't assist with that.
FINE-TUNED MODEL: Yes, we can repair your laptop.
------------------------------------------------------------

QUESTION: Can I get my laptop screen replaced?
ORIGINAL MODEL: I'm sorry, but as an AI language model, I don't have the capability to replace your lapt

In [15]:

import shutil
from google.colab import files

# Compress the saved LoRA adapter folder
shutil.make_archive(
    "techzone_lora_adapter",
    "zip",
    "techzone_lora_adapter"
)

print("LoRA adapter ZIP created successfully!")

# Download the ZIP file
files.download("techzone_lora_adapter.zip")


LoRA adapter ZIP created successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>